In [18]:
import json
import gurobipy as gp
from gurobipy import GRB, quicksum

### 학번: 2025324096
### 이름: 장서현
### Variable 선정 이유: Completion Time Variables
#### 목적함수($\sum w_j T_j$)와 제약 조건이 모두 완료시간($C_j$)을 매개로 정의되므로, 완료시간 변수를 사용하는 것이 가장 직관적이고 간결하다고 판단했다. 특히 Precedence Constraints (job $i \to j$)의 경우, $C_j \ge C_i + p_j$ 형태의 단순한 선형 부등식만으로 처리할 수 있어 구현이 효율적이다. 
### Formulation은 IDO6001_HW4.pdf (Problem 1.1)에 첨부하였습니다. 

## Dataset
### Set
- N = $\{1,2,...,n\}, j \in N$
- $prec$: Set of precedence $(i,j) \in prec$

### Parameters
- $p_j$: processing time of job $j$
- $d_j$: due date of job $j$
- $w_j$: weight of job $j$

In [19]:
with open('single_prec_dataset.json') as file:
    data = json.load(file)

print(len(data['N']))
print(data['p'])
print(data['d'])
print(data['w'])
print(data['prec'])

12
{'0': 88, '1': 91, '2': 65, '3': 22, '4': 63, '5': 13, '6': 9, '7': 70, '8': 12, '9': 16, '10': 18, '11': 62}
{'0': 104, '1': 432, '2': 476, '3': 418, '4': 431, '5': 190, '6': 285, '7': 169, '8': 221, '9': 148, '10': 169, '11': 331}
{'0': 2, '1': 7, '2': 1, '3': 5, '4': 2, '5': 5, '6': 3, '7': 5, '8': 5, '9': 5, '10': 10, '11': 2}
{'0': [11], '1': [], '2': [6, 3, 11], '3': [11, 8], '4': [0], '5': [], '6': [9], '7': [6, 4, 9, 11], '8': [], '9': [], '10': [3, 5], '11': []}


In [20]:
n = len(data['N']) # 작업의 개수

# Sets
N = [_ for _ in range(n)] # 작업 set을 리스트 자료구조를 이용해 0, 1, 2,..., n-1 까지 담아줌

# Parameters
p = [data['p'][f'{i}'] for i in N] # 작업의 가공시간 데이터를 리스트 자료구조를 이용해 만들어 줌
d = [data['d'][f'{i}'] for i in N] # 작업의 마감 기한 데이터를 리스트 자료구조를 이용해 만들어 줌
w = [data['w'][f'{i}'] for i in N] # 작업의 가중치 데이터를 리스트 자료구조를 이용해 만들어 줌

prec = [[i, j] for i in N if data['prec'][f'{i}'] != [] for j in data['prec'][f'{i}']] # ex) 작업 1 -> 작업 12, 작업 3 -> 작업 7 ...

M = sum(p)* 2 # big M

In [21]:
print(p)
print(d)
print(w)
print(prec)

[88, 91, 65, 22, 63, 13, 9, 70, 12, 16, 18, 62]
[104, 432, 476, 418, 431, 190, 285, 169, 221, 148, 169, 331]
[2, 7, 1, 5, 2, 5, 3, 5, 5, 5, 10, 2]
[[0, 11], [2, 6], [2, 3], [2, 11], [3, 11], [3, 8], [4, 0], [6, 9], [7, 6], [7, 4], [7, 9], [7, 11], [10, 3], [10, 5]]


In [22]:
for i, j in prec:
    print(i, j)

0 11
2 6
2 3
2 11
3 11
3 8
4 0
6 9
7 6
7 4
7 9
7 11
10 3
10 5


## Code

In [23]:
# Precedence Set 구성
prec_set = set((i, j) for i, j in prec)

# Unordered Pairs Set (U) 생성: i < j 이면서 선행 관계가 없는 쌍들만 추출
U = []
for i in N:
    for j in N:
        if i < j:
            if (i, j) not in prec_set and (j, i) not in prec_set:
                U.append((i, j))

In [24]:
model = gp.Model("Problem 1.1")
model.setParam("OutputFlag", 0)

# Decision Variables 정의
C = model.addVars(N, vtype=GRB.CONTINUOUS, lb=0, name="C")
T = model.addVars(N, vtype=GRB.CONTINUOUS, lb=0, name="T")
y = model.addVars(U, vtype=GRB.BINARY, name="y")

# Objective Function 정의
model.setObjective(quicksum(w[j] * T[j] for j in N), GRB.MINIMIZE)

# Constraints 정의
for j in N: model.addConstr(T[j] >= C[j] - d[j], name=f"Tardiness_{j}")
for j in N: model.addConstr(C[j] >= p[j], name=f"BasicCompletion_{j}")
for i, j in prec_set: model.addConstr(C[j] >= C[i] + p[j], name=f"Prec_{i}_{j}")
for i, j in U:
    model.addConstr(C[j] >= C[i] + p[j] - M * (1 - y[i, j]), name=f"Seq_Fwd_{i}_{j}")
    model.addConstr(C[i] >= C[j] + p[i] - M * y[i, j], name=f"Seq_Bwd_{i}_{j}")

# 최적화 실행
model.optimize()

In [25]:
# 결과 출력
if model.status == GRB.OPTIMAL:
    print(f"\nOptimal Objective Value: {model.objVal}")
    
    sorted_jobs = sorted(N, key=lambda j: C[j].X)
    print("Optimal Sequence:", sorted_jobs)
    
    print(f"{'Job':<5} | {'Completion':<12} | {'Due Date':<10} | {'Tardiness':<10}")
    print("-" * 45)
    for j in sorted_jobs:
        print(f"{j:<5} | {C[j].X:<12.1f} | {d[j]:<10} | {T[j].X:<10.1f}")


Optimal Objective Value: 1297.0
Optimal Sequence: [7, 2, 10, 6, 9, 5, 3, 8, 1, 4, 0, 11]
Job   | Completion   | Due Date   | Tardiness 
---------------------------------------------
7     | 70.0         | 169        | 0.0       
2     | 135.0        | 476        | 0.0       
10    | 153.0        | 169        | 0.0       
6     | 162.0        | 285        | 0.0       
9     | 178.0        | 148        | 30.0      
5     | 191.0        | 190        | 1.0       
3     | 213.0        | 418        | 0.0       
8     | 225.0        | 221        | 4.0       
1     | 316.0        | 432        | 0.0       
4     | 379.0        | 431        | 0.0       
0     | 467.0        | 104        | 363.0     
11    | 529.0        | 331        | 198.0     
